# `MF6ADJ` demonstration using the synthetic dewatering example

In this notebook, we will see now `mf6adj` can be used with an MODFLOW-6 version of a version of the synthetic mine dewatering example presented in White and other 2025 " Reliable Trade‐offs Between Environment and Economy: Implications for Mine Dewatering and Managed Aquifer Recharge"

In [ ]:
import os
import pathlib as pl
import platform
import shutil
import sys
from datetime import datetime

import flopy
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyemu

In [ ]:
try:
    import mf6adj
except ImportError:
    sys.path.insert(0, str(pl.Path("../").resolve()))
    import mf6adj

First we need to get the platform-specific binaries.  We have some strict control over these and they are stored at the root level in the repo in the `bin` dir.  Let's workout what path we should be using and the binary names we need:

In [ ]:
env_path = pl.Path(os.environ.get("CONDA_PREFIX", None))
assert env_path is not None, "Notebook must be run from the mf6adj Conda environment"

In [ ]:
bin_path = "bin"
exe_ext = ""
if "linux" in platform.platform().lower():
    lib_ext = ".so"
elif "darwin" in platform.platform().lower() or "macos" in platform.platform().lower():
    lib_ext = ".dylib"
else:
    bin_path = "Scripts"
    lib_ext = ".dll"
    exe_ext = ".exe"
lib_name = env_path / f"{bin_path}/libmf6{lib_ext}"
mf6_bin = env_path / f"{bin_path}/mf6{exe_ext}"

Now let's get the model files we will be using - they are stored in the autotest directory

In [ ]:
org_ws = os.path.join("synthdewater")
assert os.path.exists(org_ws)

setup a local copy of the model files.  Also copy in the binaries we need for later....

In [ ]:
ws = "synthdewater_working"
if os.path.exists(ws):
    shutil.rmtree(ws)
shutil.copytree(org_ws, ws)

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=ws)
m = sim.get_model()
X, Y = m.modelgrid.xcellcenters, m.modelgrid.ycellcenters

In [ ]:
def plot_model(k, arr, units=None, cmap="plasma", center=False, levels=None):
    vmin = None
    vmax = None

    if center:
        mx = np.nanmax(np.abs(arr))
        vmin = -1.0 * mx
        vmax = mx
        cmap = "coolwarm"

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.set_aspect("equal")
    mv = flopy.plot.PlotMapView(model=m, ax=ax)
    mv.plot_bc("WEL-dewater", label="Dewater Wells")
    mv.plot_bc("WEL-mar", label="Injection Wells")

    mv.plot_bc("DRN", color="green", label="Drain - GDE")
    mv.plot_bc("GHB", color="blue", label="GHB - regional aquifer")
    cb = ax.pcolormesh(X, Y, arr, cmap=cmap, vmin=vmin, vmax=vmax, alpha=0.5)

    plt.colorbar(cb, ax=ax, label=units)
    if levels is not None:
        c = "w"
        if center:
            c = "k"
        CS = ax.contour(X, Y, arr, levels=levels, colors=c)
        ax.clabel(CS, CS.levels, fontsize=10)

    return fig, ax

In [ ]:
fig, ax = plot_model(0, m.dis.idomain.array[0, :, :].astype(float))
_ = ax.set_title("idomain")

The pit is meant to be in the idomain==2 region

In [ ]:
fig, ax = plt.subplots()
mv = flopy.plot.PlotMapView(model=m)
mv.plot_grid(lw=0.5)
mv.plot_bc("WEL-dewater", label="Dewater Wells")
mv.plot_bc("WEL-mar", label="Injection Wells")

mv.plot_bc("DRN", color="green", label="Drain - GDE")
mv.plot_bc("GHB", color="blue", label="GHB - regional aquifer")

# mv.plot_array(gwf.dis.idomain.get_data(), color='gray', alpha=1)

There we see the GDE DRN boundary on the left, the inflow GHB on the right and the dewatering and reinjection wells.  The model has 3 stress periods: pre-development (SS), active mining (10yr transient) and closure (20year transient)

In [ ]:
fig, ax = plot_model(0, np.log10(m.npf.k.array[0, :, :]), levels=3)
_ = ax.set_title("HK")

A beautiful work of nonstationary geostats...
Run the existing model in our local workspace

In [ ]:
pyemu.os_utils.run(mf6_bin.name, cwd=ws)

Now plot some heads...

In [ ]:
labels = ["predev", "end of mining", "closure"]
hds = flopy.utils.HeadFile(os.path.join(ws, "model.hds"))
for kper, label in enumerate(labels):
    final_arr = hds.get_data(kstpkper=(0, kper))
    fig, ax = plot_model(0, final_arr[0, :, :], units="meters", levels=5)
    ax.set_title(label)

So groundwater flows from high head to low head...

The main requirement to use `Mf6Adj` is an input file that describes the performance measures.  Luckily this file has a nice modern format like other MF6 input files.  Here we are going to make these programmatically - we want to check the head in the pit at the end of the mining period and the flux to the GDE boundary in pre-development, end-of-mining and after closure


In [ ]:
drn = pd.DataFrame.from_records(m.drn.stress_period_data.array[0])
drn

In [ ]:
names = ["drn-gde-predev", "drn-gde-endmining", "drn-gde-postclosure"]
pm_fname = "prefmeas.dat"
fpm = open(os.path.join(ws, pm_fname), "w")

for kper, name in enumerate(names):
    fpm.write("begin performance_measure {0}\n".format(name))

    for kij in drn.cellid.values:
        fpm.write(
            "{0} 1 {1} {2} {3} drn-gde direct 1.0 -1.0e+30\n".format(
                kper + 1, kij[0] + 1, kij[1] + 1, kij[2] + 1
            )
        )
    fpm.write("end performance_measure\n\n")

In [ ]:
name = "pithead-endmining"

kij = (0, 49, 49)
fpm.write("begin performance_measure {0}\n".format(name))
fpm.write(
    "{0} 1 {1} {2} {3} head direct 1.0 -1.0e+30\n".format(
        kper + 1, kij[0] + 1, kij[1] + 1, kij[2] + 1
    )
)
fpm.write("end performance_measure\n\n")
fpm.close()

Ok, now we should be ready to go...the adjoint solution process requires running the model forward once and then solving for the adjoint state, which uses the forward solution components (i.e. the conductance matrix, the RHS, heads, saturation,etc). The adjoint state solution has two important characteristics:  its a linear (independent of the forward model's linearity) and it solves backward in time, starting with the last stress period - WAT?!

The adjoint solve is considerably slower than the forward solution, with most of the time being spent in the numpy sparse linear solve...#lyf

In [ ]:
bd = os.getcwd()
os.chdir(ws)

In [ ]:
forward_hdf5_name = "forward.hdf5"
start = datetime.now()

adj = mf6adj.Mf6Adj(pm_fname, lib_name, logging_level="INFO")
adj.solve_gwf(hdf5_name=forward_hdf5_name)  # solve the standard forward solution
dfsum = adj.solve_adjoint()  # solve the adjoint state for each performance measure
adj.finalize()  # release components
duration = (datetime.now() - start).total_seconds()
print("took:", duration)

In [ ]:
os.chdir(bd)

Boo ya!  done...let's see what happened...

In [ ]:
hdf5_files = [f for f in os.listdir(ws) if f.endswith("hdf5")]
hdf5_files.sort()
hdf5_files = hdf5_files[:-1]
hdf5_files

`MF6ADJ` uses the widely available HDF5 format to store information - these files hold very low-level granular information about the adjoint solution.  However the `mf6adj.solve_adjoint()` method also returns a higher-level summary of the adjoint solution.  Let's look at it first:

In [ ]:
type(dfsum)

In [ ]:
list(dfsum.keys())

In [ ]:
dfhw = dfsum["pithead-endmining"]
dfhw

those are the node-scale sensitivities to the end-of-mining pit groundwater level performance measure - some plots would be nice you say?!  Well this is most easily done with the HDF5 file itself...

In [ ]:
result_hdf = hdf5_files[-1]
hdf = h5py.File(os.path.join(ws, result_hdf), "r")
keys = list(hdf.keys())
keys.sort()
print(keys)

The "composite" group has the sensitivities of the performance measure to the model inputs summed across all adjoint solutions...

In [ ]:
grp = hdf["composite"]
plot_keys = [
    i for i in grp.keys() if len(grp[i].shape) == 3 and ("k11" in i or "ss" in i)
]
plot_keys

A simple routine to plot all these sensitivities....

In [ ]:
for pkey in plot_keys:
    arr = grp[pkey][:]
    for k, karr in enumerate(arr):
        karr[karr == 0.0] = np.nan
        fig, ax = plot_model(k, karr, center=True, levels=4)
        ax.set_title(pkey + ", layer:{0}".format(k + 1), loc="left")

sweet - that is the map of sensitvity of the end of mining pit groundwater level to HK and SS...is that pattern what you were expecting? 

Lets look at the same plots for the GDE-flux performance measures:

In [ ]:
for result_hdf in hdf5_files:
    hdf = h5py.File(os.path.join(ws, result_hdf), "r")
    grp = hdf["composite"]
    pm_name = (
        result_hdf.replace("adjoint_solution_", "")
        .split(".")[0]
        .replace("_forward", "")
    )
    for pkey in plot_keys:
        arr = grp[pkey][:]
        for k, karr in enumerate(arr):
            karr[karr == 0.0] = np.nan
            fig, ax = plot_model(k, karr, center=True, levels=4)
            ax.set_title(pm_name + ", " + pkey, loc="left")

Are those sensitivities coherent with your expectations and "hydrosense"?  Lets look at the HK array again:

In [ ]:
fig, ax = plot_model(0, np.log10(m.npf.k.array[0, :, :]), levels=3)
_ = ax.set_title("HK")

Let's also look at the sensitivity of groundwater pumping during the active mining period to the head in the pit at the end of mining:

In [ ]:
mining_grp = [key for key in keys if key.startswith("solution_kper:00001")]
assert len(mining_grp) == 1
mining_grp = mining_grp[0]
mining_grp

In [ ]:
plot_keys = [i for i in grp.keys() if len(grp[i].shape) == 3 and ("wel6_q" in i)]
plot_keys

In [ ]:
for result_hdf in hdf5_files:
    hdf = h5py.File(os.path.join(ws, result_hdf), "r")
    grp = hdf[mining_grp]
    pm_name = (
        result_hdf.replace("adjoint_solution_", "")
        .split(".")[0]
        .replace("_forward", "")
    )
    for pkey in plot_keys:
        arr = grp[pkey][:]
        for k, karr in enumerate(arr):
            karr[karr == 0.0] = np.nan
            fig, ax = plot_model(k, karr, center=True, levels=4)
            ax.set_title(pm_name + ", " + pkey, loc="left")

This is a way to understanding how groundwater extraction and reinjection during active mining influences both pit groundwater levels and the GDE flux at the end of mining and at the end of the closure period.  